# Intelligent Decision Support System for Solar PV Forecasting and Operational Management

**MSc project notebook — 15-minute-ahead plant-level AC power forecasting**

The experiment compares Ridge Regression, Random Forest and XGBoost against a persistence benchmark. Each plant is modelled separately because the two plants have different operating patterns and power scales.

### Methodological safeguards

 reported test results reflect a realistic time-series setting:

- Predictors use measurements available at the forecast issue time or earlier.
- Missing weather readings are never filled from future measurements. A short gap can only be carried forward from the most recent past reading, for a maximum of two 15-minute slots.
- Duplicate `DATE_TIME` + `SOURCE_KEY` records are checked before aggregation.
- Train, validation and test data remain in chronological order.
- A one-horizon purge is applied at split boundaries so that a training target cannot overlap the first issue time of the next block.
- `TimeSeriesSplit` also uses a one-step gap during hyperparameter tuning.
- Missing targets are never imputed.
- Preprocessing is fitted inside scikit-learn pipelines and therefore only learns from the relevant training fold.
- Model selection is based on validation RMSE. The final test block is not used for model choice.
- Persistence remains a benchmark rather than participating in the trained-model selection.
- Percentage error is reported only when generation is above a small operating threshold, avoiding unnecessary division by near-zero values from night time.

The notebook also exports the data-quality summary, chronological split information, model comparisons, diagnostic plots, feature importance, decision-support output, saved model bundles and run configuration.

## 1. Environment setup

 XGBoost is installed automatically in Colab if it is not already available. A fallback model is included for code checking outside Colab, although final project results should be produced with XGBoost available and `fastMode=False`.

In [ ]:
try:
    from IPython.display import display
except ImportError:
    def display(value):
        try:
            print(value.to_string(index=False))
        except Exception:
            print(value)

import importlib.util
import os
import subprocess
import sys

try:
    inColab = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    inColab = False

if importlib.util.find_spec("xgboost") is None and inColab:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "xgboost>=2.0,<4"]
    )

xgboostAvailable = importlib.util.find_spec("xgboost") is not None
print(f"Running in Google Colab: {inColab}")
print(f"XGBoost available: {xgboostAvailable}")


In [ ]:
import json
import math
import platform
import random
import shutil
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.ensemble import (
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance as permutationImportance
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error as meanAbsoluteError,
    mean_squared_error as meanSquaredError,
    r2_score as r2Score,
)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

if xgboostAvailable:
    from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")

randomState = 42
np.random.seed(randomState)
random.seed(randomState)

# Forecast configuration
frequencyMinutes = 15
forecastHorizonSteps = 1
validationFraction = 0.15
testFraction = 0.15
minInverterCoverage = 0.75
weatherForwardFillLimit = 2  # carry forward at most two consecutive 15-minute gaps
cvSplits = 4

# fastMode is only for checking that the notebook runs. Final results require fastMode=False.
fastMode = os.environ.get("PV_FAST_MODE", "0") == "1"

# In Colab, files can be uploaded into /content or read from Google Drive.
dataDir = Path(os.environ.get("PV_DATA_DIR", "/content" if inColab else "upload"))
outputDir = Path(
    os.environ.get(
        "PV_OUTPUT_DIR",
        "/content/pv_idss_outputs" if inColab else "pv_idss_outputs",
    )
)
outputDir.mkdir(parents=True, exist_ok=True)

expectedFiles = {
    1: {
        "generation": "Plant_1_Generation_Data.csv",
        "weather": "Plant_1_Weather_Sensor_Data.csv",
    },
    2: {
        "generation": "Plant_2_Generation_Data.csv",
        "weather": "Plant_2_Weather_Sensor_Data.csv",
    },
}

print("Python:", platform.python_version())
print("Pandas:", pd.__version__)
print("Data directory:", dataDir.resolve())
print("Output directory:", outputDir.resolve())
print("Fast mode:", fastMode)

## 2. Load and validate the four CSV files

The two generation files contain inverter-level observations, while the two weather files contain plant-level sensor readings. Before any modelling takes place, the notebook checks that all required files and columns are present and that timestamps can be parsed correctly.

In [ ]:
def requiredPaths(dataDir: Path) -> list[Path]:
    return [
        dataDir / item[fileType]
        for item in expectedFiles.values()
        for fileType in ("generation", "weather")
    ]


missingPaths = [path for path in requiredPaths(dataDir) if not path.exists()]

if missingPaths and inColab:
    from google.colab import files

    print("Upload these four files:")
    for path in missingPaths:
        print(" -", path.name)
    uploaded = files.upload()
    dataDir = Path("/content")
    missingPaths = [path for path in requiredPaths(dataDir) if not path.exists()]

if missingPaths:
    missingNames = "\n".join(f" - {path}" for path in missingPaths)
    raise FileNotFoundError(
        "The following required files were not found:\n"
        f"{missingNames}\n"
        "Set DATA_DIR to the folder containing the CSV files and rerun this cell."
    )

print("All required files were found.")


In [ ]:
generationColumns = {
    "DATE_TIME",
    "PLANT_ID",
    "SOURCE_KEY",
    "DC_POWER",
    "AC_POWER",
    "DAILY_YIELD",
    "TOTAL_YIELD",
}
weatherColumns = {
    "DATE_TIME",
    "PLANT_ID",
    "SOURCE_KEY",
    "AMBIENT_TEMPERATURE",
    "MODULE_TEMPERATURE",
    "IRRADIATION",
}


def parseDatetime(series: pd.Series) -> pd.Series:
    # Parse both DD-MM-YYYY and ISO timestamps used by the supplied files.
    text = series.astype(str).str.strip()
    parsed = pd.to_datetime(text, format="%d-%m-%Y %H:%M", errors="coerce")
    missing = parsed.isna()
    parsed.loc[missing] = pd.to_datetime(
        text.loc[missing], format="%Y-%m-%d %H:%M:%S", errors="coerce"
    )
    missing = parsed.isna()
    parsed.loc[missing] = pd.to_datetime(text.loc[missing], errors="coerce")
    return parsed


def loadAndValidateCsv(path: Path, expectedColumns: set[str]) -> pd.DataFrame:
    frame = pd.read_csv(path)

    missingColumns = expectedColumns.difference(frame.columns)
    if missingColumns:
        raise ValueError(f"{path.name} is missing columns: {sorted(missingColumns)}")

    frame["DATE_TIME"] = parseDatetime(frame["DATE_TIME"])
    if frame["DATE_TIME"].isna().any():
        badCount = int(frame["DATE_TIME"].isna().sum())
        raise ValueError(f"{path.name} contains {badCount} invalid timestamps.")

    # A repeated timestamp for the same source could silently distort the plant aggregation,
    # even when the rows are not exact duplicates. Treat that as a data-quality error.
    duplicateKeyMask = frame.duplicated(
        subset=["DATE_TIME", "SOURCE_KEY"],
        keep=False,
    )
    if duplicateKeyMask.any():
        badCount = int(duplicateKeyMask.sum())
        examples = (
            frame.loc[duplicateKeyMask, ["DATE_TIME", "SOURCE_KEY"]]
            .head(5)
            .to_dict("records")
        )
        raise ValueError(
            f"{path.name} contains {badCount} rows with repeated DATE_TIME + SOURCE_KEY "
            f"combinations. Example records: {examples}"
        )

    frame = frame.sort_values("DATE_TIME").reset_index(drop=True)
    return frame


rawData = {}
for plant, names in expectedFiles.items():
    generation = loadAndValidateCsv(
        dataDir / names["generation"], generationColumns
    )
    weather = loadAndValidateCsv(
        dataDir / names["weather"], weatherColumns
    )
    rawData[plant] = {"generation": generation, "weather": weather}

rawSummary = []
for plant, parts in rawData.items():
    for dataType, frame in parts.items():
        rawSummary.append(
            {
                "Plant": plant,
                "Dataset": dataType.title(),
                "Rows": len(frame),
                "Start": frame["DATE_TIME"].min(),
                "End": frame["DATE_TIME"].max(),
                "Missing cells": int(frame.isna().sum().sum()),
                "Exact duplicate rows": int(frame.duplicated().sum()),
                "Repeated timestamp-source rows": int(
                    frame.duplicated(subset=["DATE_TIME", "SOURCE_KEY"]).sum()
                ),
                "Source keys": int(frame["SOURCE_KEY"].nunique()),
            }
        )

rawSummaryDf = pd.DataFrame(rawSummary)
display(rawSummaryDf)

## 3. Prepare plant-level generation and weather data

Each generation file contains one row per inverter, so the inverter readings are aggregated before they are merged with weather data.

A simple sum would make plant output appear to fall whenever one or more inverter records are missing. To avoid that, full-plant power is estimated from the mean of the available inverter readings multiplied by the expected inverter count. The estimate is accepted only when at least 75% of the plant's inverters are represented. The coverage value is retained as a quality indicator rather than hidden.

### Weather gaps

 A missing weather reading can only use the latest value already observed in the past, and only for a maximum of two consecutive 15-minute slots. Longer gaps remain missing and are handled later by the training-fitted imputer inside each model pipeline. The original weather-availability flag is kept so the model can distinguish measured from filled observations.

In [ ]:
def preparePlantFrame(
    generation: pd.DataFrame,
    weather: pd.DataFrame,
    plantNumber: int,
) -> tuple[pd.DataFrame, dict]:
    expectedInverters = int(generation["SOURCE_KEY"].nunique())

    generationAgg = (
        generation.groupby("DATE_TIME")
        .agg(
            AC_POWER_RAW=("AC_POWER", "sum"),
            AC_POWER_MEAN=("AC_POWER", "mean"),
            DC_POWER_RAW=("DC_POWER", "sum"),
            DC_POWER_MEAN=("DC_POWER", "mean"),
            INVERTER_COUNT=("SOURCE_KEY", "nunique"),
        )
        .sort_index()
    )

    weatherAgg = (
        weather.groupby("DATE_TIME")
        .agg(
            AMBIENT_TEMPERATURE=("AMBIENT_TEMPERATURE", "mean"),
            MODULE_TEMPERATURE=("MODULE_TEMPERATURE", "mean"),
            IRRADIATION=("IRRADIATION", "mean"),
        )
        .sort_index()
    )

    start = min(generationAgg.index.min(), weatherAgg.index.min())
    end = max(generationAgg.index.max(), weatherAgg.index.max())
    fullIndex = pd.date_range(start, end, freq=f"{frequencyMinutes}min")

    generationAgg = generationAgg.reindex(fullIndex)
    weatherAgg = weatherAgg.reindex(fullIndex)

    frame = generationAgg.join(weatherAgg, how="outer")
    frame.index.name = "DATE_TIME"
    frame["PLANT_NUMBER"] = plantNumber
    frame["EXPECTED_INVERTERS"] = expectedInverters
    frame["INVERTER_COVERAGE"] = frame["INVERTER_COUNT"] / expectedInverters
    frame["GENERATION_AVAILABLE"] = frame["INVERTER_COUNT"].notna().astype(int)
    frame["WEATHER_AVAILABLE"] = frame[
        ["AMBIENT_TEMPERATURE", "MODULE_TEMPERATURE", "IRRADIATION"]
    ].notna().all(axis=1).astype(int)

    # Estimate full-plant power only when enough inverter records are available.
    adequateCoverage = frame["INVERTER_COVERAGE"] >= minInverterCoverage
    frame["AC_POWER"] = np.where(
        adequateCoverage,
        frame["AC_POWER_MEAN"] * expectedInverters,
        np.nan,
    )
    frame["DC_POWER"] = np.where(
        adequateCoverage,
        frame["DC_POWER_MEAN"] * expectedInverters,
        np.nan,
    )

    weatherColumns = [
        "AMBIENT_TEMPERATURE",
        "MODULE_TEMPERATURE",
        "IRRADIATION",
    ]
    # Causal treatment of short weather gaps.
    # Only a value already observed in the past may be carried forward.
    weatherMissingBeforeFill = frame[weatherColumns].isna().any(axis=1)
    frame[weatherColumns] = frame[weatherColumns].ffill(
        limit=weatherForwardFillLimit
    )
    frame["WEATHER_FILLED_FROM_PAST"] = (
        weatherMissingBeforeFill
        & frame[weatherColumns].notna().all(axis=1)
    ).astype(int)

    quality = {
        "Plant": plantNumber,
        "Expected 15-minute slots": len(fullIndex),
        "Generation slots present": int(generationAgg["INVERTER_COUNT"].notna().sum()),
        "Weather slots originally complete": int(frame["WEATHER_AVAILABLE"].sum()),
        "Weather slots filled from past": int(frame["WEATHER_FILLED_FROM_PAST"].sum()),
        "Slots with all inverters": int(
            (frame["INVERTER_COUNT"] == expectedInverters).sum()
        ),
        f"Slots with >= {minInverterCoverage:.0%} coverage": int(
            adequateCoverage.sum()
        ),
        "Expected inverters": expectedInverters,
    }
    return frame, quality


plantFrames = {}
qualityRecords = []
for plant, parts in rawData.items():
    plantFrame, quality = preparePlantFrame(
        parts["generation"], parts["weather"], plant
    )
    plantFrames[plant] = plantFrame
    qualityRecords.append(quality)

dataQualityDf = pd.DataFrame(qualityRecords)
display(dataQualityDf)
dataQualityDf.to_csv(outputDir / "data_quality_summary.csv", index=False)

## 4. Exploratory data analysis

The exploratory plots provide a quick check of the daily solar pattern, the relationship between irradiation and plant AC power, the change in average power over time and the distribution of inverter-record coverage. These plots are descriptive only; they do not influence model selection.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for plant, frame in plantFrames.items():
    sampleEnd = frame.index.min() + pd.Timedelta(days=7)
    sample = frame.loc[frame.index <= sampleEnd]
    axes[0, 0].plot(sample.index, sample["AC_POWER"], label=f"Plant {plant}", alpha=0.85)

    daylight = frame[(frame["IRRADIATION"] > 0.02) & frame["AC_POWER"].notna()]
    axes[0, 1].scatter(
        daylight["IRRADIATION"],
        daylight["AC_POWER"],
        s=10,
        alpha=0.35,
        label=f"Plant {plant}",
    )

    dailyPower = frame["AC_POWER"].resample("D").mean()
    axes[1, 0].plot(dailyPower.index, dailyPower, marker="o", label=f"Plant {plant}")

    axes[1, 1].hist(
        frame["INVERTER_COVERAGE"].dropna(),
        bins=np.linspace(0.5, 1.0, 21),
        alpha=0.55,
        label=f"Plant {plant}",
    )

axes[0, 0].set_title("Plant-level AC power: first seven days")
axes[0, 0].set_ylabel("AC power (dataset units)")
axes[0, 0].legend()

axes[0, 1].set_title("Irradiation versus AC power during daylight")
axes[0, 1].set_xlabel("Irradiation (dataset units)")
axes[0, 1].set_ylabel("AC power (dataset units)")
axes[0, 1].legend()

axes[1, 0].set_title("Daily mean AC power")
axes[1, 0].set_ylabel("Mean AC power (dataset units)")
axes[1, 0].legend()

axes[1, 1].set_title("Inverter-record coverage")
axes[1, 1].set_xlabel("Available inverter proportion")
axes[1, 1].set_ylabel("Timestamp count")
axes[1, 1].legend()

for ax in axes.flat:
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
edaPath = outputDir / "exploratory_data_analysis.png"
plt.savefig(edaPath, dpi=180, bbox_inches="tight")
plt.show()


## 5. Time-series feature engineering

The forecasting target is plant AC power one 15-minute step ahead:

\[
y_t = AC\_POWER_{t+1}
\]

Predictors are constructed from information available at the issue time: current telemetry, past lags, backward-looking rolling statistics, recent changes and deterministic time-of-day/year
No measured power or weather value from the target timestamp is included as a predictor. The target and target-quality columns are kept for modelling and audit purposes only, then explicitly excluded from the feature list.

In [ ]:
powerLags = [1, 2, 4, 8, 96]
irradiationLags = [1, 2, 4, 8]
rollingWindows = [4, 8, 16, 96]


def buildFeatureTable(frame: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    features = pd.DataFrame(index=frame.index)

    # Information available at forecast issue time t.
    features["AC_POWER_CURRENT"] = frame["AC_POWER"]
    features["DC_POWER_CURRENT"] = frame["DC_POWER"]
    features["IRRADIATION_CURRENT"] = frame["IRRADIATION"]
    features["AMBIENT_TEMPERATURE_CURRENT"] = frame["AMBIENT_TEMPERATURE"]
    features["MODULE_TEMPERATURE_CURRENT"] = frame["MODULE_TEMPERATURE"]
    features["INVERTER_COVERAGE_CURRENT"] = frame["INVERTER_COVERAGE"]
    features["WEATHER_AVAILABLE_CURRENT"] = frame["WEATHER_AVAILABLE"]

    for lag in powerLags:
        minutes = lag * frequencyMinutes
        features[f"AC_POWER_T_MINUS_{minutes}MIN"] = frame["AC_POWER"].shift(lag)
        features[f"DC_POWER_T_MINUS_{minutes}MIN"] = frame["DC_POWER"].shift(lag)

    for lag in irradiationLags:
        minutes = lag * frequencyMinutes
        features[f"IRRADIATION_T_MINUS_{minutes}MIN"] = frame["IRRADIATION"].shift(lag)

    for window in rollingWindows:
        minutes = window * frequencyMinutes
        minimumPeriods = max(2, window // 2)
        features[f"AC_POWER_ROLLING_MEAN_{minutes}MIN"] = frame["AC_POWER"].rolling(
            window, min_periods=minimumPeriods
        ).mean()
        features[f"AC_POWER_ROLLING_STD_{minutes}MIN"] = frame["AC_POWER"].rolling(
            window, min_periods=minimumPeriods
        ).std()
        features[f"IRRADIATION_ROLLING_MEAN_{minutes}MIN"] = frame[
            "IRRADIATION"
        ].rolling(window, min_periods=minimumPeriods).mean()

    features["AC_POWER_CHANGE_15MIN"] = frame["AC_POWER"].diff(1)
    features["AC_POWER_CHANGE_30MIN"] = frame["AC_POWER"].diff(2)
    features["IRRADIATION_CHANGE_15MIN"] = frame["IRRADIATION"].diff(1)
    features["MODULE_AMBIENT_TEMP_DIFFERENCE"] = (
        frame["MODULE_TEMPERATURE"] - frame["AMBIENT_TEMPERATURE"]
    )

    forecastTimes = features.index + pd.Timedelta(
        minutes=frequencyMinutes * forecastHorizonSteps
    )
    hourDecimal = forecastTimes.hour + forecastTimes.minute / 60.0
    dayOfYear = forecastTimes.dayofyear

    features["FORECAST_HOUR_SIN"] = np.sin(2 * np.pi * hourDecimal / 24)
    features["FORECAST_HOUR_COS"] = np.cos(2 * np.pi * hourDecimal / 24)
    features["FORECAST_DAY_SIN"] = np.sin(2 * np.pi * dayOfYear / 365.25)
    features["FORECAST_DAY_COS"] = np.cos(2 * np.pi * dayOfYear / 365.25)
    features["FORECAST_IS_DAYLIGHT"] = (frame["IRRADIATION"] > 0.02).astype(int)

    # Target and audit-only columns are excluded from the predictors.
    features["TARGET_AC_POWER"] = frame["AC_POWER"].shift(-forecastHorizonSteps)
    features["TARGET_INVERTER_COVERAGE"] = frame["INVERTER_COVERAGE"].shift(
        -forecastHorizonSteps
    )
    features["TARGET_TIME"] = forecastTimes

    excluded = {
        "TARGET_AC_POWER",
        "TARGET_INVERTER_COVERAGE",
        "TARGET_TIME",
    }
    featureColumns = [column for column in features.columns if column not in excluded]

    # Simple leakage checks: target/audit columns must never enter the predictors,
    # and every target timestamp must be later than its forecast issue time.
    assert excluded.isdisjoint(featureColumns)
    validTargetTimes = features["TARGET_TIME"].notna()
    assert (
        features.loc[validTargetTimes, "TARGET_TIME"]
        > features.index[validTargetTimes]
    ).all()

    return features, featureColumns


featureTables = {}
modelTables = {}
featureColumnsByPlant = {}

for plant, frame in plantFrames.items():
    featureTable, featureColumns = buildFeatureTable(frame)
    validTarget = featureTable["TARGET_AC_POWER"].notna()
    validCurrent = featureTable["AC_POWER_CURRENT"].notna()
    validFutureCoverage = (
        featureTable["TARGET_INVERTER_COVERAGE"] >= minInverterCoverage
    )
    modelTable = featureTable.loc[
        validTarget & validCurrent & validFutureCoverage
    ].copy()

    featureTables[plant] = featureTable
    modelTables[plant] = modelTable
    featureColumnsByPlant[plant] = featureColumns

    print(
        f"Plant {plant}: {len(modelTable):,} modelling rows, "
        f"{len(featureColumns)} predictors, "
        f"{modelTable.index.min()} to {modelTable.index.max()}"
    )

## 6. Chronological train-validation-test split

The earliest observations form the training block, followed by validation and then the final test block. The intended proportions remain approximately 70% / 15% / 15%.

to remove any boundary overlap, the last affected row is purged from training and validation where necessary.

The code then asserts that:

- the latest training target is earlier than the first validation issue time, and
- the latest validation target is earlier than the first test issue time.

This makes the hold-out boundaries consistent with the one-step-ahead forecasting setup.

In [ ]:
def chronologicalSplit(table: pd.DataFrame) -> dict[str, pd.DataFrame]:
    nRows = len(table)
    trainEnd = int(nRows * (1 - validationFraction - testFraction))
    validationEnd = int(nRows * (1 - testFraction))

    if trainEnd < 100 or validationEnd <= trainEnd or validationEnd >= nRows:
        raise ValueError("Insufficient data for the configured chronological split.")

    train = table.iloc[:trainEnd].copy()
    validation = table.iloc[trainEnd:validationEnd].copy()
    test = table.iloc[validationEnd:].copy()

    validationStart = validation.index.min()
    testStart = test.index.min()

    # Purge any row whose target reaches into the next block.
    train = train.loc[train["TARGET_TIME"] < validationStart].copy()
    validation = validation.loc[validation["TARGET_TIME"] < testStart].copy()

    if train.empty or validation.empty or test.empty:
        raise ValueError("Purging left an empty chronological split.")

    assert train.index.max() < validation.index.min() < test.index.min()
    assert train["TARGET_TIME"].max() < validation.index.min()
    assert validation["TARGET_TIME"].max() < test.index.min()

    return {
        "train": train,
        "validation": validation,
        "test": test,
    }


splitRecords = []
splitsByPlant = {}
for plant, table in modelTables.items():
    splits = chronologicalSplit(table)
    splitsByPlant[plant] = splits
    for splitName, splitFrame in splits.items():
        splitRecords.append(
            {
                "Plant": plant,
                "Split": splitName.title(),
                "Rows": len(splitFrame),
                "Issue-time start": splitFrame.index.min(),
                "Issue-time end": splitFrame.index.max(),
                "Target-time end": splitFrame["TARGET_TIME"].max(),
                "Boundary-safe": True,
            }
        )

splitSummaryDf = pd.DataFrame(splitRecords)
display(splitSummaryDf)
splitSummaryDf.to_csv(outputDir / "chronological_split_summary.csv", index=False)

## 7. Evaluation functions

The notebook reports MAE, RMSE, NRMSE and R² for the full evaluation block.

NRMSE is calculated as RMSE divided by the observed target range in the relevant evaluation block.

In [ ]:
def regressionMetrics(
    yTrue: np.ndarray | pd.Series,
    yPred: np.ndarray | pd.Series,
    daylightThreshold: float,
) -> dict[str, float]:
    yTrueArray = np.asarray(yTrue, dtype=float)
    yPredArray = np.clip(np.asarray(yPred, dtype=float), 0, None)

    mae = meanAbsoluteError(yTrueArray, yPredArray)
    rmse = math.sqrt(meanSquaredError(yTrueArray, yPredArray))
    observedRange = float(np.nanmax(yTrueArray) - np.nanmin(yTrueArray))
    nrmse = 100 * rmse / observedRange if observedRange > 0 else np.nan
    r2 = r2Score(yTrueArray, yPredArray)

    daylightMask = yTrueArray > daylightThreshold
    if daylightMask.any():
        daylightMae = meanAbsoluteError(
            yTrueArray[daylightMask], yPredArray[daylightMask]
        )
        daylightRmse = math.sqrt(
            meanSquaredError(
                yTrueArray[daylightMask], yPredArray[daylightMask]
            )
        )
        daylightMape = 100 * np.mean(
            np.abs(
                (yTrueArray[daylightMask] - yPredArray[daylightMask])
                / yTrueArray[daylightMask]
            )
        )
    else:
        daylightMae = daylightRmse = daylightMape = np.nan

    return {
        "MAE": float(mae),
        "RMSE": float(rmse),
        "NRMSE_RANGE_%": float(nrmse),
        "R2": float(r2),
        "DAYLIGHT_MAE": float(daylightMae),
        "DAYLIGHT_RMSE": float(daylightRmse),
        "DAYLIGHT_MAPE_%": float(daylightMape),
    }


## 8. Model construction and time-series-aware tuning

The experiment compares four forecasting approaches:

1. **Persistence** — the next 15-minute value is assumed to equal the current AC power.
2. **Ridge Regression** — a regularised linear benchmark.
3. **Random Forest** — a nonlinear bagged-tree model.
4. **XGBoost** — a regularised boosted-tree model.

Hyperparameters are tuned using expanding-window `TimeSeriesSplit` folds inside the training block. A one-step gap is included between the training and validation portions of each fold so that the one-step-ahead target does not touch the next fold boundary.

After tuning, each candidate is assessed on the chronological validation block. The machine-learning model with the lowest validation RMSE is selected for that plant. Persistence remains a benchmark and only after model selection is complete is the final untouched test block evaluated.

In [ ]:
def buildSearches() -> dict[str, object]:
    cv = TimeSeriesSplit(n_splits=cvSplits, gap=forecastHorizonSteps)

    ridgePipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", StandardScaler()),
            ("model", Ridge()),
        ]
    )
    ridgeSearch = GridSearchCV(
        ridgePipeline,
        {"model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]},
        scoring="neg_root_mean_squared_error",
        cv=cv,
        n_jobs=-1,
        refit=True,
    )

    forestPipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            (
                "model",
                RandomForestRegressor(
                    random_state=randomState,
                    n_jobs=-1,
                ),
            ),
        ]
    )
    forestParameters = {
        "model__n_estimators": [200, 400, 600],
        "model__max_depth": [None, 8, 12, 16],
        "model__min_samples_leaf": [1, 2, 4, 8],
        "model__max_features": [0.6, 0.8, 1.0],
    }
    forestSearch = RandomizedSearchCV(
        forestPipeline,
        forestParameters,
        n_iter=4 if fastMode else 16,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        random_state=randomState,
        n_jobs=-1,
        refit=True,
    )

    if xgboostAvailable:
        boostingModel = XGBRegressor(
            objective="reg:squarederror",
            eval_metric="rmse",
            tree_method="hist",
            random_state=randomState,
            n_jobs=-1,
            verbosity=0,
        )
        boostingParameters = {
            "model__n_estimators": [200, 350, 500, 700],
            "model__max_depth": [2, 3, 4, 5, 6],
            "model__learning_rate": [0.02, 0.04, 0.06, 0.1],
            "model__subsample": [0.7, 0.85, 1.0],
            "model__colsample_bytree": [0.7, 0.85, 1.0],
            "model__min_child_weight": [1, 3, 5, 8],
            "model__reg_alpha": [0.0, 0.01, 0.1, 0.5],
            "model__reg_lambda": [1.0, 3.0, 5.0, 10.0],
        }
        boostingName = "XGBoost"
        boostingIterations = 6 if fastMode else 24
    else:
        boostingModel = HistGradientBoostingRegressor(
            random_state=randomState,
            early_stopping=True,
        )
        boostingParameters = {
            "model__max_iter": [150, 250, 400],
            "model__learning_rate": [0.03, 0.05, 0.1],
            "model__max_leaf_nodes": [15, 31, 63],
            "model__l2_regularization": [0.0, 0.1, 1.0, 5.0],
        }
        boostingName = "Histogram Gradient Boosting (XGBoost fallback)"
        boostingIterations = 4 if fastMode else 12

    boostingPipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("model", boostingModel),
        ]
    )
    boostingSearch = RandomizedSearchCV(
        boostingPipeline,
        boostingParameters,
        n_iter=boostingIterations,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        random_state=randomState,
        n_jobs=-1,
        refit=True,
    )

    return {
        "Ridge Regression": ridgeSearch,
        "Random Forest": forestSearch,
        boostingName: boostingSearch,
    }


def fitAndSelectModel(
    plant: int,
    splits: dict[str, pd.DataFrame],
    featureColumns: list[str],
) -> dict:
    train = splits["train"]
    validation = splits["validation"]
    test = splits["test"]

    xTrain = train[featureColumns]
    yTrain = train["TARGET_AC_POWER"]
    xValidation = validation[featureColumns]
    yValidation = validation["TARGET_AC_POWER"]
    xTest = test[featureColumns]
    yTest = test["TARGET_AC_POWER"]

    capacityEstimate = float(yTrain.quantile(0.99))
    daylightThreshold = max(capacityEstimate * 0.01, 1e-9)

    validationRecords = []
    fittedCandidates = {}

    persistenceValidation = np.clip(
        validation["AC_POWER_CURRENT"].to_numpy(), 0, None
    )
    persistenceMetrics = regressionMetrics(
        yValidation, persistenceValidation, daylightThreshold
    )
    validationRecords.append(
        {"Plant": plant, "Model": "Persistence", **persistenceMetrics}
    )

    for modelName, search in buildSearches().items():
        print(f"Plant {plant}: tuning {modelName}...")
        search.fit(xTrain, yTrain)
        validationPrediction = np.clip(search.predict(xValidation), 0, None)
        metrics = regressionMetrics(
            yValidation, validationPrediction, daylightThreshold
        )
        validationRecords.append(
            {"Plant": plant, "Model": modelName, **metrics}
        )
        fittedCandidates[modelName] = {
            "estimator": search.best_estimator_,
            "parameters": search.best_params_,
            "cv_rmse": float(-search.best_score_),
            "validation_prediction": validationPrediction,
        }

    validationTable = pd.DataFrame(validationRecords).sort_values("RMSE")

    # Persistence is a benchmark, not a trainable ML model.
    bestModelName = min(
        fittedCandidates,
        key=lambda name: float(
            validationTable.loc[
                validationTable["Model"] == name, "RMSE"
            ].iloc[0]
        ),
    )
    selectedCandidate = fittedCandidates[bestModelName]
    validationActual = yValidation.to_numpy()
    validationResiduals = (
        validationActual - selectedCandidate["validation_prediction"]
    )

    def residualInterval(residuals: np.ndarray) -> dict[str, float]:
        return {
            "lower_5pct": float(np.quantile(residuals, 0.05)),
            "upper_95pct": float(np.quantile(residuals, 0.95)),
        }

    daylightValidation = validationActual > daylightThreshold
    residualQuantiles = {"all": residualInterval(validationResiduals)}
    residualQuantiles["daylight"] = (
        residualInterval(validationResiduals[daylightValidation])
        if daylightValidation.sum() >= 30
        else residualQuantiles["all"]
    )
    residualQuantiles["night"] = (
        residualInterval(validationResiduals[~daylightValidation])
        if (~daylightValidation).sum() >= 30
        else residualQuantiles["all"]
    )

    trainValidation = pd.concat([train, validation]).sort_index()
    finalModel = clone(selectedCandidate["estimator"])
    finalModel.fit(
        trainValidation[featureColumns],
        trainValidation["TARGET_AC_POWER"],
    )

    testPrediction = np.clip(finalModel.predict(xTest), 0, None)
    testMetrics = regressionMetrics(yTest, testPrediction, daylightThreshold)
    persistenceTest = np.clip(test["AC_POWER_CURRENT"].to_numpy(), 0, None)
    persistenceTestMetrics = regressionMetrics(
        yTest, persistenceTest, daylightThreshold
    )

    testRecords = [
        {
            "Plant": plant,
            "Model": bestModelName,
            **testMetrics,
        },
        {
            "Plant": plant,
            "Model": "Persistence",
            **persistenceTestMetrics,
        },
    ]

    return {
        "plant": plant,
        "feature_columns": featureColumns,
        "validation_table": validationTable,
        "selected_model_name": bestModelName,
        "selected_parameters": selectedCandidate["parameters"],
        "selected_cv_rmse": selectedCandidate["cv_rmse"],
        "final_model": finalModel,
        "capacity_estimate": capacityEstimate,
        "daylight_threshold": daylightThreshold,
        "residual_quantiles": residualQuantiles,
        "test_frame": test,
        "test_prediction": testPrediction,
        "persistence_test_prediction": persistenceTest,
        "test_results": pd.DataFrame(testRecords),
        "trained_through": trainValidation.index.max(),
    }

## 9. Train, select and compare the models

This section runs the complete model-selection procedure for each plant, saves the validation and test comparison tables, and prints the selected model family. Final submission runs should use `fastMode=False`.

In [ ]:
experimentResults = {}

for plant in sorted(splitsByPlant):
    print("=" * 72)
    print(f"TRAINING PLANT {plant}")
    experimentResults[plant] = fitAndSelectModel(
        plant,
        splitsByPlant[plant],
        featureColumnsByPlant[plant],
    )
    print(f"Selected model for Plant {plant}: {experimentResults[plant]['selected_model_name']}")
    display(experimentResults[plant]["validation_table"])

validationComparisonDf = pd.concat(
    [result["validation_table"] for result in experimentResults.values()],
    ignore_index=True,
)
testComparisonDf = pd.concat(
    [result["test_results"] for result in experimentResults.values()],
    ignore_index=True,
)

validationComparisonDf.to_csv(
    outputDir / "validation_model_comparison.csv", index=False
)
testComparisonDf.to_csv(outputDir / "test_model_comparison.csv", index=False)

print("Final untouched test results")
display(testComparisonDf.sort_values(["Plant", "RMSE"]))


## 10. Test-set visualisation and residual analysis

The diagnostic plots compare actual and predicted AC power, show actual-versus-predicted behaviour, and inspect residuals against the predicted level and as a distribution. 

In [ ]:
def plotTestDiagnostics(result: dict) -> Path:
    plant = result["plant"]
    test = result["test_frame"]
    actual = test["TARGET_AC_POWER"].to_numpy()
    predicted = result["test_prediction"]
    residuals = actual - predicted
    targetTimes = pd.to_datetime(test["TARGET_TIME"])

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    axes[0, 0].plot(targetTimes, actual, label="Actual", linewidth=1.5)
    axes[0, 0].plot(targetTimes, predicted, label="Forecast", linewidth=1.2)
    axes[0, 0].set_title(f"Plant {plant}: actual versus 15-minute forecast")
    axes[0, 0].set_ylabel("AC power (dataset units)")
    axes[0, 0].legend()

    axes[0, 1].scatter(actual, predicted, s=16, alpha=0.5)
    upper = max(float(np.max(actual)), float(np.max(predicted)))
    axes[0, 1].plot([0, upper], [0, upper], "--", color="black", linewidth=1)
    axes[0, 1].set_title("Actual versus predicted")
    axes[0, 1].set_xlabel("Actual AC power")
    axes[0, 1].set_ylabel("Predicted AC power")

    axes[1, 0].scatter(predicted, residuals, s=16, alpha=0.5)
    axes[1, 0].axhline(0, color="black", linestyle="--", linewidth=1)
    axes[1, 0].set_title("Residuals versus predictions")
    axes[1, 0].set_xlabel("Predicted AC power")
    axes[1, 0].set_ylabel("Residual: actual - predicted")

    axes[1, 1].hist(residuals, bins=35, alpha=0.8)
    axes[1, 1].axvline(0, color="black", linestyle="--", linewidth=1)
    axes[1, 1].set_title("Residual distribution")
    axes[1, 1].set_xlabel("Residual")
    axes[1, 1].set_ylabel("Count")

    for ax in axes.flat:
        ax.tick_params(axis="x", rotation=25)

    plt.tight_layout()
    path = outputDir / f"plant_{plant}_test_diagnostics.png"
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    return path


diagnosticPaths = [
    plotTestDiagnostics(result) for result in experimentResults.values()
]


## 11. Model interpretation

Permutation importance is calculated after model selection. Each feature is shuffled while the others are left unchanged, and the resulting increase in RMSE is used as a common measure of model reliance.

This is useful for comparing the selected Random Forest and XGBoost models on the same basis. It should not be interpreted as a causal explanation, particularly where predictors are correlated.

In [ ]:
def featureImportanceTable(result: dict, topN: int = 20) -> pd.DataFrame:
    modelPipeline = result["final_model"]
    test = result["test_frame"]
    featureColumns = result["feature_columns"]
    xTest = test[featureColumns]
    yTest = test["TARGET_AC_POWER"]

    # Permutation importance works consistently for every possible selected model
    # and is calculated only after final testing.
    sampleSize = min(len(xTest), 800)
    xSample = xTest.iloc[-sampleSize:]
    ySample = yTest.iloc[-sampleSize:]
    permutation = permutationImportance(
        modelPipeline,
        xSample,
        ySample,
        n_repeats=8 if not fastMode else 3,
        random_state=randomState,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
    )
    importance = pd.DataFrame(
        {
            "Feature": featureColumns,
            "Importance": permutation.importances_mean,
            "Importance_std": permutation.importances_std,
        }
    ).sort_values("Importance", ascending=False)
    return importance.head(topN)


importanceTables = {}
fig, axes = plt.subplots(1, 2, figsize=(17, 8))

for ax, (plant, result) in zip(axes, experimentResults.items()):
    importance = featureImportanceTable(result)
    importanceTables[plant] = importance
    importance.to_csv(
        outputDir / f"plant_{plant}_feature_importance.csv", index=False
    )
    plotFrame = importance.sort_values("Importance", ascending=True)
    ax.barh(plotFrame["Feature"], plotFrame["Importance"])
    ax.set_title(f"Plant {plant}: permutation feature importance")
    ax.set_xlabel("Increase in RMSE after feature permutation")

plt.tight_layout()
importancePath = outputDir / "feature_importance.png"
plt.savefig(importancePath, dpi=180, bbox_inches="tight")
plt.show()


## 12. Intelligent decision-support output

The forecasting model produces a number; the IDSS layer turns that number into an interpretable advisory record containing:

- the plant and selected model,
- issue and target timestamps,
- current and forecast AC power,
- an empirical residual-based interval,
- inverter coverage,
- an operating status, and
- a short recommendation.


In [ ]:
def createDecisionRecord(plant: int, result: dict) -> dict:
    featureTable = featureTables[plant]
    featureColumns = result["feature_columns"]

    eligible = featureTable[
        (featureTable["AC_POWER_CURRENT"].notna())
        & (
            featureTable["INVERTER_COVERAGE_CURRENT"]
            >= minInverterCoverage
        )
    ]
    latest = eligible.iloc[-1]
    xLatest = latest[featureColumns].to_frame().T

    rawPredicted = float(
        np.clip(result["final_model"].predict(xLatest)[0], 0, None)
    )
    current = float(latest["AC_POWER_CURRENT"])
    capacity = result["capacity_estimate"]
    lowGenerationThreshold = capacity * 0.01
    # Use observed irradiation as the night/day guard.

    forecastIsDaylight = bool(
        float(latest["IRRADIATION_CURRENT"]) > 0.02
    )
    intervalRegime = "daylight" if forecastIsDaylight else "night"
    interval = result["residual_quantiles"][intervalRegime]

    if forecastIsDaylight:
        predicted = rawPredicted
        lower = min(
            predicted,
            max(0.0, predicted + interval["lower_5pct"]),
        )
        upper = max(
            predicted,
            predicted + interval["upper_95pct"],
        )
    else:
        # PV generation is physically unavailable at night. 
        predicted = 0.0
        lower = 0.0
        upper = 0.0

    if current > max(capacity * 0.01, 1e-9):
        changePercent = 100 * (predicted - current) / current
    else:
        changePercent = np.nan

    if predicted <= capacity * 0.01:
        status = "Very low or zero generation expected"
        recommendation = "Treat solar contribution as unavailable for the next interval."
    elif np.isfinite(changePercent) and changePercent <= -20:
        status = "Sharp generation reduction expected"
        recommendation = (
            "Prepare stored or alternative supply and defer discretionary demand if possible."
        )
    elif np.isfinite(changePercent) and changePercent >= 20:
        status = "Generation increase expected"
        recommendation = (
            "Consider charging storage or scheduling flexible demand during the increase."
        )
    else:
        status = "Generation expected to remain broadly stable"
        recommendation = "Continue normal operation while monitoring the next update."

    coverage = float(latest["INVERTER_COVERAGE_CURRENT"])
    if coverage < 0.9:
        status += "; reduced telemetry coverage"
        recommendation += " Verify missing inverter telemetry before operational action."

    return {
        "Plant": plant,
        "Model": result["selected_model_name"],
        "Forecast issued at": latest.name,
        "Forecast timestamp": latest.name
        + pd.Timedelta(minutes=frequencyMinutes * forecastHorizonSteps),
        "Current AC power": current,
        "Forecast AC power": predicted,
        "90% empirical lower bound": lower,
        "90% empirical upper bound": upper,
        "Interval regime": intervalRegime,
        "Expected change %": changePercent,
        "Current inverter coverage": coverage,
        "Status": status,
        "Recommendation": recommendation,
    }


decisionSupportDf = pd.DataFrame(
    [
        createDecisionRecord(plant, result)
        for plant, result in experimentResults.items()
    ]
)

containmentMask = (
    (decisionSupportDf["90% empirical lower bound"]
     <= decisionSupportDf["Forecast AC power"])
    & (decisionSupportDf["Forecast AC power"]
       <= decisionSupportDf["90% empirical upper bound"])
)
nightMask = decisionSupportDf["Interval regime"].eq("night")
assert containmentMask.all(), "Every interval must contain its point forecast."
assert (
    decisionSupportDf.loc[
        nightMask,
        [
            "Forecast AC power",
            "90% empirical lower bound",
            "90% empirical upper bound",
        ],
    ]
    .eq(0.0)
    .all()
    .all()
), "Night forecasts and interval bounds must all be zero."

display(decisionSupportDf)
decisionSupportDf.to_csv(
    outputDir / "latest_decision_support_forecast.csv", index=False
)
print(
    "Decision-support export passed interval containment and night-zero checks."
)

## 13. Save models, metadata and reproducibility outputs

The selected model for each plant is saved together with the exact feature order, forecast horizon, coverage rule, capacity estimate, residual quantiles, training cut-off and selected parameters.

The notebook also writes the run configuration to JSON and creates a ZIP archive of the output directory. These files make it possible to trace the reported results back to a specific modelling setup.

In [ ]:
modelFiles = []
metadataRecords = []
# This CV score comes only from expanding-window folds inside the training block.
# Hold-out validation RMSE remains separate and selects the final model family.

for plant, result in experimentResults.items():
    modelBundle = {
        "project": "Intelligent Decision Support System for Solar PV Forecasting",
        "plant": plant,
        "model_name": result["selected_model_name"],
        "model": result["final_model"],
        "feature_columns": result["feature_columns"],
        "forecast_horizon_steps": forecastHorizonSteps,
        "frequency_minutes": frequencyMinutes,
        "minimum_inverter_coverage": minInverterCoverage,
        "capacity_estimate": result["capacity_estimate"],
        "residual_quantiles": result["residual_quantiles"],
        "trained_through": str(result["trained_through"]),
        "selected_parameters": result["selected_parameters"],
        "created_utc": datetime.now(timezone.utc).isoformat(),
    }
    modelPath = outputDir / f"plant_{plant}_solar_pv_forecaster.joblib"
    joblib.dump(modelBundle, modelPath)
    modelFiles.append(modelPath)

    metadataRecords.append(
        {
            "Plant": plant,
            "Selected model": result["selected_model_name"],
            "Time-series CV RMSE (training folds)": result["selected_cv_rmse"],
            "Trained through": result["trained_through"],
            "Feature count": len(result["feature_columns"]),
            "Model file": modelPath.name,
        }
    )

metadataDf = pd.DataFrame(metadataRecords)
display(metadataDf)
metadataDf.to_csv(outputDir / "model_metadata.csv", index=False)

runConfiguration = {
    "frequency_minutes": frequencyMinutes,
    "forecast_horizon_steps": forecastHorizonSteps,
    "validation_fraction": validationFraction,
    "test_fraction": testFraction,
    "minimum_inverter_coverage": minInverterCoverage,
    "weather_forward_fill_limit": weatherForwardFillLimit,
    "purge_gap_steps": forecastHorizonSteps,
    "time_series_cv_splits": cvSplits,
    "random_state": randomState,
    "xgboost_available": xgboostAvailable,
    "fast_mode": fastMode,
    "created_utc": datetime.now(timezone.utc).isoformat(),
}
with open(outputDir / "run_configuration.json", "w", encoding="utf-8") as file:
    json.dump(runConfiguration, file, indent=2)

archivePath = shutil.make_archive(
    str(outputDir.parent / "pv_idss_outputs"), "zip", root_dir=outputDir
)
print("Saved results archive:", archivePath)

downloadResults = False
if downloadResults and inColab:
    from google.colab import files

    files.download(archivePath)

## 14. Reload a saved model

This short check confirms that a saved model bundle can be loaded again and that it still contains the fitted preprocessing/model pipeline and the exact predictor order needed for later inference.

In [ ]:
exampleBundle = joblib.load(modelFiles[0])
print("Loaded plant:", exampleBundle["plant"])
print("Loaded model:", exampleBundle["model_name"])
print("Forecast horizon:", exampleBundle["frequency_minutes"], "minutes")
print("Number of required predictors:", len(exampleBundle["feature_columns"]))


## 15. Interpretation 

The final report shows:

- data coverage, missing timestamps and inverter-coverage findings for both plants;
- the exact chronological split dates and the purged boundary rows;
- validation results for persistence, Ridge Regression, Random Forest and XGBoost;
- untouched test results for the selected model and persistence benchmark;
- full-period and active-generation error metrics;
- residual plots and evidence of where errors become larger;
- permutation feature importance;
- the 34-day observation window as a limitation;
- the empirical nature of the forecast interval;
- the fact that this is a 15-minute telemetry-based forecast rather than a day-ahead forecast using numerical weather prediction;
- the advisory nature of the decision rules



### Research foundation

- Essam, Y. et al. (2022), *Investigating photovoltaic solar power output forecasting using machine learning algorithms*, Engineering Applications of Computational Fluid Mechanics, 16(1), 2002–2034. https://doi.org/10.1080/19942060.2022.2126528
- Gaboitaolelwe, J. et al. (2023), *Machine Learning Based Solar Photovoltaic Power Forecasting: A Review and Comparison*, IEEE Access, 11. https://doi.org/10.1109/ACCESS.2023.3270041
- Scott, C., Ahsan, M. and Albarbar, A. (2023), *Machine learning for forecasting a photovoltaic (PV) generation system*, Energy, 278, 127807. https://doi.org/10.1016/j.energy.2023.127807
- Deng, Y. and Lumley, T. (2024), *Multiple Imputation Through XGBoost*, Journal of Computational and Graphical Statistics, 33(2), 352–363. https://doi.org/10.1080/10618600.2023.2252501
- Luo, C. et al. (2025), *Photovoltaic system modeling and forecasting techniques: A survey*, Engineering Applications of Artificial Intelligence, 162, 112516. https://doi.org/10.1016/j.engappai.2025.112516
- Parczyk, P. and Burduk, R. (2025), *Short- and medium-term horizon photovoltaic power forecasting based on terrestrial solar radiation features*, Engineering Applications of Artificial Intelligence, 162, 112668. https://doi.org/10.1016/j.engappai.2025.112668
- Rchid, A.M. et al. (2025), *Solar photovoltaic power generation using machine learning considering weather conditions: A case study of Biret, Mauritania*, Engineering Applications of Artificial Intelligence, 162, 112621. https://doi.org/10.1016/j.engappai.2025.112621
- Kumar, N. et al. (2026), *Machine learning-based predictive modeling for solar photovoltaic power forecasting: Analyzing the influence of module temperature*, Next Energy, 11, 100589. https://doi.org/10.1016/j.nxener.2026.100589